# **Sem2 application con RHC dataset**
Dataset info can be found here: https://search.r-project.org/CRAN/refmans/ATbounds/html/RHC.html.

As a first step towards we want to show that sem2 can be applied to answer causal questions about right heart catheterization (RHC) treatment on cardiovascular patients.

RHC is a diagnostic procedure for directly measuring cardiac function in critically ill patients. The dependent variable is 1 if a patient survived after 30 days of admission, 0 if a patient died within 30 days. The treatment variable is 1 if RHC was applied within 24 hours of admission, and 0 otherwise. The sample size was **n = 5735**, and **2184 patients** were treated with RHC. Connors et al. (1996) used a propensity score matching approach to study the efficacy of RHC, using data from the observational study called SUPPORT (Murphy and Cluff, 1990). Many authors used this dataset subsequently. The 72 covariates are constructed, following Hirano and Imbens (2001).





We draw the Directed Acyclic Graph (DAG) of the problem and we use it to build the sem2 model.

In [ ]:
#IMPORTS
from sem2.model.models import SEM2Discrete
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
#from engression.loss_func import energy_loss, energy_loss_two_sample
import matplotlib.pyplot as plt
import time

In [ ]:
import pandas as pd
from google.colab import files
import io

print("Upload your CSV file here:")  # the rhc.csv file
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}"')
  # Assuming the first uploaded file is the one we want
  df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
  print("CSV file loaded into DataFrame 'df'.")
  print(df.head())
  break # Process only the first uploaded file


Upload your CSV file here:


Saving rhc.csv to rhc.csv
User uploaded file "rhc.csv"
CSV file loaded into DataFrame 'df'.
   Unnamed: 0               cat1           cat2   ca  sadmdte  dschdte  \
0           1               COPD            NaN  Yes    11142  11151.0   
1           2      MOSF w/Sepsis            NaN   No    11799  11844.0   
2           3  MOSF w/Malignancy  MOSF w/Sepsis  Yes    12083  12143.0   
3           4                ARF            NaN   No    11146  11183.0   
4           5      MOSF w/Sepsis            NaN   No    12035  12037.0   

    dthdte  lstctdte death  cardiohx  ...  meta  hema  seps  trauma  ortho  \
0      NaN     11382    No         0  ...    No    No    No      No     No   
1  11844.0     11844   Yes         1  ...    No    No   Yes      No     No   
2      NaN     12400    No         0  ...    No    No    No      No     No   
3  11183.0     11182   Yes         0  ...    No    No    No      No     No   
4  12037.0     12036   Yes         0  ...    No    No    No      No     N

In [ ]:
df.head()

,Unnamed: 0,cat1,cat2,ca,sadmdte,dschdte,dthdte,lstctdte,death,cardiohx,...,meta,hema,seps,trauma,ortho,adld3p,urin1,race,income,ptid
0,1,COPD,NaN,Yes,11142,11151.0,NaN,11382,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,white,Under $11k,5
1,2,MOSF w/Sepsis,NaN,No,11799,11844.0,11844.0,11844,1.0,1,...,0.0,0.0,1.0,0.0,0.0,NaN,1437.0,white,Under $11k,7
2,3,MOSF w/Malignancy,MOSF w/Sepsis,Yes,12083,12143.0,NaN,12400,0.0,0,...,0.0,0.0,0.0,0.0,0.0,NaN,599.0,white,$25-$50k,9
3,4,ARF,NaN,No,11146,11183.0,11183.0,11182,1.0,0,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,white,$11-$25k,10
4,5,MOSF w/Sepsis,NaN,No,12035,12037.0,12037.0,12036,1.0,0,...,0.0,0.0,0.0,0.0,0.0,NaN,64.0,white,Under $11k,11


In [ ]:
df[['swang1', 'dth30']]

,swang1,dth30
0,No RHC,0.0
1,RHC,0.0
2,RHC,0.0
3,No RHC,0.0
4,RHC,1.0
...,...,...
5730,No RHC,0.0
5731,No RHC,0.0
5732,No RHC,0.0
5733,No RHC,1.0


## **Data Preprocessing**

First step will be getting data ready to use. In particular, we aim to build the DAG from the rhc dataset and then use sem2's design on it.

N.B.: Category variables must be converted into numerical ones.

In [ ]:
for col in df.columns:
    if df[col].dtype == 'object':
        # Convert 'Yes'/'No' columns to 1/0 (float to allow for NaNs)
        if df[col].isin(['Yes', 'No']).all() or df[col].isin(['Yes', 'No', None, np.nan]).all():
            df[col] = df[col].map({'Yes': 1, 'No': 0}).astype(float)

# Select only numerical columns (including those converted from 'Yes'/'No')
df_numeric = df.select_dtypes(include=[np.number])

# IMPUTATION HERE
# Fill NaN values in numerical columns, e.g., with the mean
df_numeric = df_numeric.fillna(df_numeric.mean())

# build numeric dataset
var_names = df_numeric.columns.values.tolist()
dataset = torch.tensor(df_numeric.to_numpy()).to(dtype = torch.float32)
batch_size = 25
dataloader = DataLoader(TensorDataset(dataset), batch_size = batch_size, shuffle = True)
print(var_names)

['Unnamed: 0', 'sadmdte', 'dschdte', 'dthdte', 'lstctdte', 'death', 'cardiohx', 'chfhx', 'dementhx', 'psychhx', 'chrpulhx', 'renalhx', 'liverhx', 'gibledhx', 'malighx', 'immunhx', 'transhx', 'amihx', 'age', 'edu', 'surv2md1', 'das2d3pc', 't3d30', 'dth30', 'aps1', 'scoma1', 'meanbp1', 'wblc1', 'hrt1', 'resp1', 'temp1', 'pafi1', 'alb1', 'hema1', 'bili1', 'crea1', 'sod1', 'pot1', 'paco21', 'ph1', 'wtkilo1', 'dnr1', 'resp', 'card', 'neuro', 'gastr', 'renal', 'meta', 'hema', 'seps', 'trauma', 'ortho', 'adld3p', 'urin1', 'ptid']


In [ ]:
from sem2.default_configs import MLPConfig

In [ ]:
# confounder variables
confounder_cols = ['cat1','ca','cardiohx','chfhx','dementhx','psychhx','chrpulhx','renalhx',
             'liverhx','gibledhx','malighx','immunhx','transhx','amihx','age','sex','edu',
             'surv2md1','das2d3pc','aps1','scoma1','meanbp1','wblc1','hrt1','resp1','temp1',
             'pafi1','alb1','hema1','bili1','crea1','sod1','pot1','paco21','ph1','wtkilo1',
             'dnr1','ninsclas','resp','card','neuro','gastr','renal','meta','hema','seps',
             'trauma','ortho','race','income']

# final dataset with confounder + swang1 'treatment' + dth30 'outcome'
d = df[confounder_cols + ['swang1', 'dth30']].dropna().copy()
print(f'nr of patients (after dropna): {len(d)}')  # expected: 5735 (no NA in this set)

nr of patients (after dropna): 5735


In [ ]:
d['cat1'].unique()

array(['COPD', 'MOSF w/Sepsis', 'MOSF w/Malignancy', 'ARF', 'CHF', 'Coma',
       'Cirrhosis', 'Lung Cancer', 'Colon Cancer'], dtype=object)

In [ ]:
import re
# treatment and outcome variables' conversion -> 0/1
d['swang1'] = (d['swang1'] == 'RHC').astype(int)
d['dth30'] = (d['dth30'] == 'Yes').astype(int)

# 0-1 confounders Yes/No -> 0/1
binary_yn = ['card','neuro','gastr','renal','meta','hema','seps','trauma','ortho','dnr1','resp']
for c in binary_yn:
    d[c] = (d[c] == 'Yes').astype(int)

d['sex'] = (d['sex'] == 'Male').astype(int)

# readable income rages before one-hot encoding
d['income'] = d['income'].replace({
    '> $50k': 'Over_50k',
    '$25-$50k': '25_50k',
    '$11-$25k': '11_25k',
    'Under $11k': 'Under_11k',
})

# Multi-category confounders -> dummy (drop_first = reference category)
multicat = ['cat1', 'ca', 'ninsclas', 'race', 'income']
d = pd.get_dummies(d, columns=multicat, drop_first=True, dtype=int)

# clear names
def sanitize(name: str) -> str:
    """Converts a column name into a valid identifier for SEM2 formulas"""
    name = re.sub(r'[^0-9a-zA-Z_]', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    if name[0].isdigit():
        name = 'v_' + name
    return name

d.columns = [sanitize(c) for c in d.columns]
print(f'Total number of columns (expanded confounders + treatment + outcome): {d.shape[1]}')

Total number of columns (expanded confounders + treatment + outcome): 67


We also standardize continuous confounders to better compare them when using SEM2.

In [ ]:
# %cd sem2-examples # better not

[Errno 2] No such file or directory: 'sem2-examples'
/content


In [ ]:
# separate continuous confounders (to be standardized) from binary/dummy variables (already encoded as 0/1)
already_binary = [c for c in d.columns if sorted(d[c].unique().tolist()) in ([0, 1], [0], [1])]
continuous_cols = [c for c in d.columns if c not in already_binary and c not in ['swang1', 'dth30']]
binary_cols = [c for c in already_binary if c not in ['swang1', 'dth30']]

print(f'Continuous confounders: {len(continuous_cols)}')
print(f'Binary/dummy confounders: {len(binary_cols)}')
print(f'Total confounders: {len(continuous_cols) + len(binary_cols)}')

# Standardize continuous confounders using z-score normalization
# Binary/dummy variables are kept in their original 0/1 encoding
for c in continuous_cols:
    d[c] = (d[c] - d[c].mean()) / d[c].std()

# save the prepared dataset for downstream SEM2 analysis
d.to_csv('rhc_prepared.csv', index=False)
print('SAVED rhc_prepared.csv')

Continuous confounders: 21
Binary/dummy confounders: 44
Total confounders: 65
SAVED rhc_prepared.csv


In [ ]:
#len(d.columns)

67

In [ ]:
from google.colab import files

files.download('rhc_prepared.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **DAG**

Now we build the Directed Acyclic Graph (DAG).

Model specification: This step constructs the sem2 model formula using the variables identified in the prepared RHC dataset. Continuous confounders are assigned standard *normal distributions*, while binary confounders and the treatment are modeled as *Bernoulli variables*. The outcome is then defined as a *function of the treatment and all confounders*, specifying the parent structure of the causal graph.

**Important:** *This is a generative specification for synthetic data. The model does not learn the underlying distributions or causal relationships from the original RHC dataset. Instead, the RHC dataset is used as a realistic template for defining the variables and graph structure, while the distributions and causal relationships are explicitly specified in the sem2 model.*

In [ ]:
# Define the generative distributions for continuous confounders
# Continuous variables are modeled using a standard normal distribution
exo_lines = [f'{c} ~ Normal(0,1)' for c in continuous_cols]
# Define the generative distributions for binary confounders
# Binary variables are modeled using a Bernoulli distribution with p = 0.5
exo_lines += [f'{c} ~ Bernoulli(0.5)' for c in binary_cols]
# Define the treatment as a binary variable with equal probability of treatment assignment
exo_lines.append('swang1 ~ Bernoulli(0.5)')

# Collect all variables that directly influence the outcome
all_parents = ['swang1'] + continuous_cols + binary_cols

# Define the outcome as a function of the treatment and all confounders
# This specifies the parent structure of the causal graph
outcome_line = f'dth30 <- d({", ".join(all_parents)})'
model_formula = '; '.join(exo_lines) + '; ' + outcome_line

print(f'Number of parents of dth30: {len(all_parents)}')
print(f'Formula length: {len(model_formula)} characters')

# Save the generated sem2 model formula to a text file
with open('rhc_formula.txt', 'w') as f:
    f.write(model_formula)

Number of parents of dth30: 66
Formula length: 2417 characters


In [ ]:
from google.colab import files

files.download('rhc_formula.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Workflow Reminder: RHC dataset → variable selection → DAG/generative model specification → MLP architecture definition → SEM2 model



In [ ]:
# Define the NN architecture for the binary outcome dth30
# The MLP takes all treatment and confounder variables as inputs and
# uses two hidden layers to model the outcome
node_cfgs = {
    'dth30': MLPConfig(
        input_dim=len(all_parents),  # Number of treatment and confounder variables
        hidden_dims=[64, 32],         # Initial hidden-layer architecture
        output_dim=2,                 # Two output classes for the binary outcome
        dropout=0.0,                 # No dropout regularization
        activation='relu',           # ReLU activation function
    )
}

# Build the SEM2 discrete model using the generative formula and MLP configuration
model = SEM2Discrete(model_formula, node_cfgs=node_cfgs)

In [ ]:
print('Number of nodes in the DAG:', len(model.topological_order))
print('Last node (should be dth30):', model.topological_order[-1])
print('N_conf (expected: 2):', model.N_conf)
print('Output dimensions (expected: [2]):', model.output_dims)

Number of nodes in the DAG: 67
Last node (should be dth30): dth30
N_conf (expected: 2): 2
Output dimensions (expected: [2]): [2]


## **Data Loader & Model Training**


In [ ]:
#### copied from: https://github.com/causal-design/sem2/blob/main/sem2/operators/discrete_energy_loss.py
import torch
import torch.nn.functional as F

class DiscreteEnergyLoss:
    def __init__(self, p: float, states: torch.Tensor, device, unbiased=True):
        """
        p: norm power
        states: tensor of shape [num_states, state_dim] containing all possible states
                Example binary state: tensor([[0, 0],
                        [0, 1],
                        [1, 0],
                        [1, 1]])
        unbiased: whether to use unbiased estimator for finite sample size correction. Unbiased can be negative.
        """
        self.unbiased = unbiased
        self.states = states.to(device)
        self.num_states = states.shape[0]

        # Dynamically generate distance matrix D_mat
        # D_mat[i,j] = ||states[i] - states[j]||^p
        self.D_mat = self._compute_distance_matrix(states, p, device)

    def _compute_distance_matrix(self, states, p, device):
        """Compute pairwise distance matrix between states."""
        num_states = states.shape[0]
        D_mat = torch.zeros((num_states, num_states), dtype=torch.float32, device=device)

        for i in range(num_states):
            for j in range(num_states):
                # Euclidean distance raised to power p
                dist = torch.norm(states[i].float() - states[j].float(), p=2) ** p
                D_mat[i, j] = dist

        return D_mat

    def idx_to_onehot(self, input_tensor):
        """Convert input tensor to one-hot encoding based on self.states.
            input_tensor: shape = [bs, n_samples, n_nodes]
                         where each element represents a state configuration
        Returns:
            onehot: shape = [bs, n_samples, num_states]
                   where num_states = self.states.shape[0]
        """
        # Get dimensions
        bs, n_samples, n_nodes = input_tensor.shape
        num_states = self.states.shape[0]

        # Flatten input for easier comparison
        input_flat = input_tensor.view(-1, n_nodes)  # [bs*n_samples, n_nodes]

        # Vectorized comparison: find matching indices in self.states
        # Expand dimensions for broadcasting: input_flat [bs*n_samples, 1, n_nodes], states [1, num_states, n_nodes]
        input_expanded = input_flat.unsqueeze(1)  # [bs*n_samples, 1, n_nodes]
        states_expanded = self.states.unsqueeze(0)  # [1, num_states, n_nodes]

        # Find exact matches for each configuration
        matches = (input_expanded == states_expanded).all(dim=2)  # [bs*n_samples, num_states]

        # Get the index of the matching state for each input configuration
        indices = torch.argmax(matches.float(), dim=1)  # [bs*n_samples]

        # Convert to one-hot encoding
        onehot = F.one_hot(indices, num_classes=num_states).float()

        # Reshape back to original batch structure
        onehot = onehot.view(bs, n_samples, num_states)

        return onehot

    def _energy_components(self, p_hat, x_true=None, p_true_mean=None):
        """Compute shared components: pred_err and unbiased pair_pred_dist.

        Args:
            p_hat: [bs, n_samples, n_states]
            x_true: [bs, n_samples, n_nodes] (optional, used if p_true_mean not given)
            p_true_mean: [bs, n_states] (optional, precomputed)
        Returns:
            (pred_err, pair_pred_dist, p_true_mean, p_true)
                p_true is None if p_true_mean was precomputed.
        """
        bs, n_samples, dim = p_hat.shape
        precomputed = p_true_mean is not None
        p_true = None

        if not precomputed:
            p_true = self.idx_to_onehot(x_true.long())
            p_true_mean = p_true.mean(dim=1)

        p_hat_mean = p_hat.mean(dim=1)

        # Cross distance: true vs pred
        prob_mat = p_true_mean.unsqueeze(-1) @ p_hat_mean.unsqueeze(1)
        pred_err = (prob_mat * self.D_mat.unsqueeze(0)).mean(dim=0).sum()

        # Pairwise pred distance
        prob_mat = p_hat_mean.unsqueeze(-1) @ p_hat_mean.unsqueeze(1)
        pair_pred_dist = (prob_mat * self.D_mat.unsqueeze(0)).mean(dim=0).sum()

        # Bias correction
        if self.unbiased:
            n_tot, n_corr = n_samples / (n_samples - 1), 1 / (n_samples * (n_samples - 1))
            correct_hat = torch.einsum("bni,ij,bnj->bn", p_hat, self.D_mat, p_hat)
            correct_hat = correct_hat.mean(dim=0).sum()
            pair_pred_dist = pair_pred_dist * n_tot - correct_hat * n_corr

        return pred_err, pair_pred_dist, p_true_mean, p_true

    def energy_score(self, x_true, p_hat, p_true_mean=None):
        """Energy score: E(|x_h - x_true|) - 1/2 E(|x_hi - x_hj|)

        Args:
            x_true: [bs, n_samples, n_nodes] iid samples from true distribution.
            p_hat:  [bs, n_samples, n_states] predicted probabilities.
            p_true_mean: [bs, n_states] optional precomputed mean.
        Returns:
            (loss, log_dict)
        """
        pred_err, pair_pred_dist, _, _ = self._energy_components(
            p_hat, x_true=x_true, p_true_mean=p_true_mean
        )
        loss = pred_err - pair_pred_dist / 2
        return loss, {"estimated-true distance": pred_err,
                      "estimated-estimated distance": pair_pred_dist}

    def energy_dist(self, p_hat, x_true=None, p_true_mean=None):
        """Energy distance: E|x_h - x_true| - 1/2 E|x_hi - x_hj| - 1/2 E|x_truei - x_truej|

        Args:
            p_hat:  [bs, n_samples, n_states] predicted probabilities.
            x_true: [bs, n_samples, n_nodes] samples (optional if p_true_mean given).
            p_true_mean: [bs, n_states] optional precomputed mean.
        Returns:
            (loss, log_dict)
        """
        pred_err, pair_pred_dist, p_true_mean, p_true = self._energy_components(
            p_hat, x_true=x_true, p_true_mean=p_true_mean
        )

        # Pairwise true distance
        prob_mat = p_true_mean.unsqueeze(-1) @ p_true_mean.unsqueeze(1)
        pair_true_dist = (prob_mat * self.D_mat.unsqueeze(0)).mean(dim=0).sum()

        if self.unbiased and p_true is not None:
            n_samples = p_hat.shape[1]
            n_tot, n_corr = n_samples / (n_samples - 1), 1 / (n_samples * (n_samples - 1))
            correct_pred = torch.einsum("bni,ij,bnj->bn", p_true, self.D_mat, p_true)
            correct_pred = correct_pred.mean(dim=0).sum()
            pair_true_dist = pair_true_dist * n_tot - correct_pred * n_corr

        loss = pred_err - pair_pred_dist / 2 - pair_true_dist / 2
        return loss, {"estimated-true distance": pred_err,
                      "estimated-estimated distance": pair_pred_dist,
                      "true-true distance": pair_true_dist}

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Stati possibili per l'esito binario dth30
states_dth30 = torch.tensor([[0.0], [1.0]], device=device)

# Inizializzatore della Loss (p=2 indica la distanza euclidea al quadrato)
criterion = DiscreteEnergyLoss(p=2.0, states=states_dth30, device=device, unbiased=True)

# **!!!HELP!!!!**

My understanding: the discrete sampling step seems to break the autograd graph, so loss.backward() cannot propagate gradients back to the MLP.

Training with sem2 discrete:

*  Which SEM2 class should I use to train a discrete outcome node such as dth30? SEM2Discrete seems to use a discrete sampling strategy that does not support backpropagation during training. The warning suggests using a different SEM2 class/sampling strategy.
* How should I obtain the predicted probabilities/logits for dth30 without sampling the discrete node? My goal is to obtain \(P(dth30 |mid parents)\) directly and use these probabilities as input to the DiscreteEnergyLoss.
* Is DiscreteEnergyLoss intended to be used directly for training the discrete outcome in this way? In particular, should the loss receive the predicted probability distribution rather than sampled discrete outcomes?
* I currently repeat the predicted probability distribution 100 times using .repeat(), but these are not independent samples. Is this the correct way to construct the input for the loss? Should unbiased=True be used in this setting? Since I am currently repeating the same prediction rather than generating independent samples, I am not sure whether the finite-sample unbiased correction is appropriate.
* More generally, what is the intended training pipeline for a discrete node in SEM2? I would like to make sure that my implementation follows the training procedure used in the original sem2 examples.






In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 30
model.to(device)

d_ordered = d[model.topological_order]
tensor_data = torch.tensor(d_ordered.values, dtype=torch.float32)
dataloader = DataLoader(TensorDataset(tensor_data), batch_size=128, shuffle=True)

dth30_idx = model.topological_order.index('dth30')

# Recuperiamo le colonne dei genitori di 'dth30'
parent_indices = [model.topological_order.index(p) for p in all_parents if p in model.topological_order]

model.train()
for epoch in range(epochs):
    total_loss = 0.0
    for batch in dataloader:
        x_batch = batch[0].to(device)
        optimizer.zero_grad()

        # 1. Estrai i genitori dal batch
        x_parents = x_batch[:, parent_indices]  # Shape: [batch_size, n_parents]
        x_true_dth30 = x_batch[:, dth30_idx].unsqueeze(-1)  # Shape: [batch_size, 1]

        # 2. Ottieni i logits dal nodo
        logits_dth30 = model.node_dict['dth30']((x_parents,))

        # 3. CONVERSIONE A FLOAT32 prima di Softmax
        logits_float = logits_dth30.float()
        p_hat_single = torch.softmax(logits_float, dim=-1)

        # 4. Adatta la forma per la DiscreteEnergyLoss
        N_SAMPLES_FOR_LOSS = 100
        p_hat_for_loss = p_hat_single.unsqueeze(1).repeat(1, N_SAMPLES_FOR_LOSS, 1)
        x_true_for_loss = x_true_dth30.unsqueeze(1).repeat(1, N_SAMPLES_FOR_LOSS, 1)

        # 5. Energy Loss differenziabile & Backpropagation
        loss, _ = criterion.energy_score(x_true=x_true_for_loss, p_hat=p_hat_for_loss)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(dataloader):.4f}")


/usr/local/lib/python3.13/dist-packages/sem2/model/components/nodes.py:293: UserWarning: Drawing sample for discrete nodes does not support backpropagation during training. 
 To train discrete nodes, use another sampling strategy, via a different SEM2 class. 
  warnings.warn("Drawing sample for discrete nodes does not support backpropagation during training. \n"


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

# ***Do*-intervention & ATE estimate**


